# Automatic Modulation Classification - Colab Notebook

This notebook trains a Baseline CNN or 1D ResNet on RadioML 2016.10A and reports overall accuracy, accuracy vs SNR, and confusion matrix.

## 1. Runtime

In Google Colab, select **Runtime > Change runtime type > T4 GPU** before running the notebook.

In [ ]:
import os, random, pickle, time, json
from copy import deepcopy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. Dataset Path

Upload `RML2016.10a_dict.pkl` to Colab, or put it in Google Drive and set `DATA_PATH` below.

In [ ]:
# Option A: upload the pickle directly to /content and use this path
DATA_PATH = '/content/RML2016.10a_dict.pkl'

# Option B: Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/RML2016.10a_dict.pkl'

assert Path(DATA_PATH).exists(), f'Dataset not found: {DATA_PATH}'

## 3. Load RadioML 2016.10A

The pickle is a dictionary with keys like `(modulation_name, snr)` and values shaped like `[1000, 2, 128]`.

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def load_radioml_2016a(data_path):
    print('Loading:', data_path)
    with open(data_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    mods = sorted({key[0] for key in data.keys()})
    snrs = sorted({int(key[1]) for key in data.keys()})
    mod_to_idx = {m: i for i, m in enumerate(mods)}

    X_list, y_list, snr_list = [], [], []
    for mod in mods:
        for snr in snrs:
            block = np.asarray(data[(mod, snr)], dtype=np.float32)
            X_list.append(block)
            y_list.append(np.full(len(block), mod_to_idx[mod], dtype=np.int64))
            snr_list.append(np.full(len(block), snr, dtype=np.int64))

    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    snr = np.concatenate(snr_list)
    print('X:', X.shape, 'y:', y.shape, 'snr:', snr.shape)
    print('Classes:', mods)
    return X, y, snr, mods

set_seed(42)
X, y, snr, classes = load_radioml_2016a(DATA_PATH)

## 4. Train / Validation / Test Split

In [ ]:
def make_dataset(X, y, snr, idx):
    return TensorDataset(
        torch.tensor(X[idx], dtype=torch.float32),
        torch.tensor(y[idx], dtype=torch.long),
        torch.tensor(snr[idx], dtype=torch.long),
    )

idx = np.arange(len(X))
stratify_key = np.array([f'{label}_{snr_value}' for label, snr_value in zip(y, snr)])
train_idx, temp_idx, _, temp_key = train_test_split(
    idx, stratify_key, test_size=0.30, random_state=42, stratify=stratify_key
)
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, temp_key, test_size=0.50, random_state=42, stratify=temp_key
)

train_ds = make_dataset(X, y, snr, train_idx)
val_ds = make_dataset(X, y, snr, val_idx)
test_ds = make_dataset(X, y, snr, test_idx)
print(len(train_ds), len(val_ds), len(test_ds))

## 5. Models

In [ ]:
def to_channel_first(x):
    if x.shape[1] == 2:
        return x
    if x.shape[2] == 2:
        return x.transpose(1, 2)
    raise ValueError(f'Cannot infer channel dimension: {x.shape}')

class BaselineCNN(nn.Module):
    def __init__(self, num_classes, dropout=0.2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(2, 64, kernel_size=7, padding=3), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(256, num_classes))

    def forward(self, x):
        return self.classifier(self.features(to_channel_first(x)))

class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.shortcut(x))

class ResNet1D(nn.Module):
    def __init__(self, num_classes, base_channels=64, dropout=0.2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(2, base_channels, 7, padding=3, bias=False),
            nn.BatchNorm1d(base_channels),
            nn.ReLU(inplace=True),
        )
        self.layer1 = nn.Sequential(ResidualBlock1D(base_channels, base_channels), ResidualBlock1D(base_channels, base_channels))
        self.layer2 = nn.Sequential(ResidualBlock1D(base_channels, base_channels * 2, stride=2), ResidualBlock1D(base_channels * 2, base_channels * 2))
        self.layer3 = nn.Sequential(ResidualBlock1D(base_channels * 2, base_channels * 4, stride=2), ResidualBlock1D(base_channels * 4, base_channels * 4))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(base_channels * 4, num_classes)

    def forward(self, x):
        x = to_channel_first(x)
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(self.dropout(x))

def build_model(model_name, num_classes):
    if model_name == 'baseline_cnn':
        return BaselineCNN(num_classes)
    if model_name == 'resnet1d':
        return ResNet1D(num_classes)
    raise ValueError(model_name)

## 6. Training Functions

In [ ]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, total_correct, total_count = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for xb, yb, _ in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            logits = model(xb)
            loss = criterion(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            total_correct += (logits.argmax(1) == yb).sum().item()
            total_count += xb.size(0)

    return total_loss / total_count, total_correct / total_count

def fit(model, train_loader, val_loader, epochs=10, lr=1e-3, weight_decay=1e-4, patience=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_state = deepcopy(model.state_dict())
    best_val_loss = float('inf')
    best_epoch = 0
    wait = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        print(f'Epoch {epoch:02d}/{epochs} | train_acc={train_acc:.4f} val_acc={val_acc:.4f} train_loss={train_loss:.4f} val_loss={val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if patience > 0 and wait >= patience:
                print('Early stopping')
                break

    model.load_state_dict(best_state)
    history['best_epoch'] = best_epoch
    history['best_val_loss'] = best_val_loss
    return model, history

def predict(model, loader):
    model.eval()
    y_true, y_pred, snrs = [], [], []
    with torch.no_grad():
        for xb, yb, sb in loader:
            logits = model(xb.to(device))
            y_true.append(yb.numpy())
            y_pred.append(logits.argmax(1).cpu().numpy())
            snrs.append(sb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred), np.concatenate(snrs)

## 7. Choose Experiment Settings

In [ ]:
MODEL_NAME = 'resnet1d'   # 'baseline_cnn' or 'resnet1d'
EPOCHS = 10
BATCH_SIZE = 512
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 5

pin_memory = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=pin_memory)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin_memory)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin_memory)

model = build_model(MODEL_NAME, num_classes=len(classes)).to(device)
print(model.__class__.__name__)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 8. Train

In [ ]:
start = time.perf_counter()
model, history = fit(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY, patience=PATIENCE)
elapsed = time.perf_counter() - start
print('Training time seconds:', round(elapsed, 2))

## 9. Training Curves

In [ ]:
epochs = np.arange(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, history['train_loss'], label='train')
axes[0].plot(epochs, history['val_loss'], label='val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[1].plot(epochs, history['train_acc'], label='train')
axes[1].plot(epochs, history['val_acc'], label='val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
plt.show()

## 10. Test Accuracy and SNR Analysis

In [ ]:
test_loss, test_acc = run_epoch(model, test_loader)
print('Test loss:', test_loss)
print('Test accuracy:', test_acc)

y_true, y_pred, test_snrs = predict(model, test_loader)
snr_scores = {}
for s in sorted(np.unique(test_snrs)):
    mask = test_snrs == s
    snr_scores[int(s)] = float(np.mean(y_true[mask] == y_pred[mask]))

for s, acc in snr_scores.items():
    print(f'{s:>3} dB: {acc * 100:5.1f}%')

plt.figure(figsize=(8, 4))
plt.plot(list(snr_scores.keys()), list(snr_scores.values()), marker='o')
plt.xlabel('SNR (dB)')
plt.ylabel('Accuracy')
plt.title('Accuracy vs SNR')
plt.grid(alpha=0.3)
plt.show()

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(classes)))
plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap='Blues')
plt.colorbar(fraction=0.046, pad=0.04)
plt.xticks(np.arange(len(classes)), classes, rotation=45, ha='right')
plt.yticks(np.arange(len(classes)), classes)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 12. Save Results

In [ ]:
out_dir = Path('/content/amc_results')
out_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), out_dir / f'{MODEL_NAME}_best_model.pt')

summary = {
    'model': MODEL_NAME,
    'test_loss': float(test_loss),
    'test_accuracy': float(test_acc),
    'accuracy_by_snr': {str(k): v for k, v in snr_scores.items()},
    'classes': classes,
    'best_epoch': history['best_epoch'],
    'training_time_sec': elapsed,
}
with open(out_dir / f'{MODEL_NAME}_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved to:', out_dir)